# Qwen2.5 SFT Expanded · Spider 1.0 test

Сравнение Qwen2.5-0.5B и Qwen2.5-1.5B с сохранёнными LoRA. Полный тест содержит 2 147 вопросов и 40 SQLite-баз. Подготовка промптов, генерация и оценщик v3 зафиксированы на коммите ветки `participant-2/evaluation`. Контрольный результат команды для 0.5B — 994/2 147 (46,30%).

1. Подключите через **Add Input** приватный Dataset с обновлённым `kaggle_sft_spider_test.py` и обоими ZIP адаптеров.
2. Включите **GPU T4** и **Internet**. Откройте [инструкцию Kaggle](https://www.kaggle.com/docs/notebooks), если нужно найти эти настройки.
3. Запустите по порядку ячейки до Smoke. Затем выставьте `RUN_FULL = True` в последней ячейке и запустите её. Файлы сохраняются в `/kaggle/working/sft_spider_test/<RUN_NAME>/`.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

INPUT_ROOT = Path("/kaggle/input")
RUN_NAME = "expanded_v3"
RUN_REFERENCE_AUDIT = False
RESUME_FROM = None  # Или Path("/kaggle/input/.../full") с папкой checkpoints.
WORK = Path("/kaggle/temp/sft_spider_test")
OUTPUT = Path("/kaggle/working/sft_spider_test") / RUN_NAME

WORK.mkdir(parents=True, exist_ok=True)
ENV = {**os.environ, "CUDA_VISIBLE_DEVICES": "0", "PYTHONUNBUFFERED": "1"}

def run(*args, cwd=None):
    subprocess.run([str(arg) for arg in args], check=True, cwd=cwd, env=ENV)

### Подготовка Spider test

Код берётся из зафиксированного коммита GitHub. Подготовка проверяет SHA‑256 архива, состав выборки и каждый эталонный запрос. Можно приложить оригинальный `spider_data.zip` отдельным входом. По умолчанию он скачивается из зеркала проекта.

In [ ]:
runners = sorted(INPUT_ROOT.rglob("kaggle_sft_spider_test.py"))
if len(runners) != 1:
    raise RuntimeError(f"Нужен один kaggle_sft_spider_test.py в Kaggle Input. Найдено {len(runners)}.")

commit = "1bf6018f71a58551ca6f66077c7a1839a68f1f88"
repo = WORK / "evaluator"
if not (repo / ".git").is_dir():
    run("git", "init", repo)
head = subprocess.run(["git", "rev-parse", "--verify", "HEAD"], cwd=repo, capture_output=True, text=True)
if head.stdout.strip() != commit:
    run("git", "fetch", "--depth=1", "https://github.com/Pavel-V418/BaseModel-vs-SFT.git", commit, cwd=repo)
    run("git", "checkout", "--detach", "FETCH_HEAD", cwd=repo)
shutil.copyfile(runners[0], repo / "src" / "kaggle_sft_spider_test.py")

archives = sorted(Path("/kaggle/input").rglob("spider_data.zip"))
if len(archives) > 1:
    raise RuntimeError("В Kaggle Input найдено несколько spider_data.zip.")
spider = archives[0] if archives else WORK / "spider_data.zip"
if not spider.is_file():
    partial = spider.with_suffix(".part")
    run("curl", "--fail", "--location", "--retry", "3", "--max-time", "480", "--output", partial,
        "https://huggingface.co/datasets/HAL-9001/spider-databases/resolve/main/spider_data.zip")
    partial.replace(spider)

def evaluate(phase):
    resume_args = ["--resume-from", RESUME_FROM] if phase == "full" and RESUME_FROM else []
    run(sys.executable, "-m", "src.kaggle_sft_spider_test", "--phase", phase,
        "--spider-archive", spider, "--input-root", INPUT_ROOT,
        "--work-dir", WORK, "--output-root", OUTPUT, *resume_args, cwd=repo)

evaluate("audit" if RUN_REFERENCE_AUDIT else "prepare")

### Smoke · первые 24 вопроса

Эта ячейка устанавливает версии Transformers, PEFT, Accelerate и PyYAML из опубликованного окружения. PyTorch берётся из среды Kaggle. Smoke проверяет загрузку обоих адаптеров, память GPU, генерацию и оценку SQL. Сводка сохраняется в `smoke/comparison.csv`.

In [ ]:
run(sys.executable, "-m", "pip", "install", "--quiet",
    "transformers==4.57.6", "peft==0.18.0", "accelerate==1.15.0", "pyyaml==6.0.3")
evaluate("smoke")
print((OUTPUT / "smoke" / "comparison.csv").read_text(encoding="utf-8"))

### Full · все 2 147 вопросов

После успешного smoke выставьте `RUN_FULL = True`. Полный прогон сохраняет контрольные точки каждые 128 вопросов. В той же сессии повторный запуск продолжит с сохранённых частей. Для новой сессии приложите сохранённый Kaggle Output и выставьте `RESUME_FROM` в ячейке настроек. После завершения сохраните версию ноутбука и скачайте файлы из **Output**.

In [ ]:
RUN_FULL = False
if RUN_FULL:
    evaluate("full")
    print((OUTPUT / "full" / "comparison.csv").read_text(encoding="utf-8"))
else:
    print("Установите RUN_FULL = True после успешного smoke.")